In [ ]:
!pip -q install -U transformers datasets accelerate sentencepiece
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
from torch.optim import AdamW

import torch
import torch.nn as nn

from torch.utils.data import Dataset, DataLoader


from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

from transformers import (
    AutoTokenizer,
    AutoModelForMultipleChoice
)

from tqdm.auto import tqdm



In [ ]:
train = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv')
test = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv')
sample = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv')

**EDA**

In [ ]:
print(train.columns)
train.head()
train.shape

In [ ]:
train.head()

In [ ]:
display(train.dtypes)

In [ ]:
train.info()

In [ ]:
print(test.shape)
print(test.columns.tolist())

In [ ]:
display(test.head())

In [ ]:
test.info()

In [ ]:
train.isnull().sum()

In [ ]:
print(train.duplicated().sum())

print(train["prompt"].duplicated().sum())

In [ ]:
duplicate_prompts = train[train["prompt"].duplicated(keep=False)]

display(
    duplicate_prompts[
        ["prompt", "A", "B", "C", "D", "E", "answer"]
    ].sort_values("prompt").head(10)
)

In [ ]:
train["answer"].value_counts()

In [ ]:
plt.figure(figsize=(7,5))

sns.countplot(
    data=train,
    x="answer",
    order=["A","B","C","D","E"]
)

plt.title("Distribution of Target Classes")
plt.xlabel("Correct Answer")
plt.ylabel("Count")

plt.show()

print(train["answer"].value_counts())

print("\nPercentage")
print((train["answer"].value_counts(normalize=True)*100).round(2))

In [ ]:
train["prompt_length"] = train["prompt"].str.split().str.len()

train["prompt_length"].describe()

In [ ]:
plt.figure(figsize=(8,5))

sns.histplot(
    train["prompt_length"],
    bins=30,
    kde=True
)

plt.title("Prompt Length Distribution")

plt.xlabel("Number of Words")

plt.ylabel("Frequency")

plt.show()

In [ ]:
for option in ["A","B","C","D","E"]:
    train[f"{option}_length"] = train[option].str.split().str.len()

In [ ]:
train[
    [
        "A_length",
        "B_length",
        "C_length",
        "D_length",
        "E_length"
    ]
].describe()

In [ ]:
avg_lengths = []

for option in ["A","B","C","D","E"]:
    avg_lengths.append(train[f"{option}_length"].mean())

plt.bar(
    ["A","B","C","D","E"],
    avg_lengths
)

plt.ylabel("Average Words")

plt.title("Average Length of Options")

plt.show()

In [ ]:
words = []

for sentence in train["prompt"]:

    words.extend(sentence.lower().split())

print("Vocabulary Size:", len(set(words)))

In [ ]:
counter = Counter(words)

counter.most_common(20)

In [ ]:
common = counter.most_common(20)

word = [x[0] for x in common]
count = [x[1] for x in common]

plt.figure(figsize=(10,5))

plt.bar(word,count)

plt.xticks(rotation=90)

plt.title("Most Frequent Words")

plt.show()

In [ ]:
train.loc[
    train["prompt_length"].idxmax(),
    "prompt"
]

In [ ]:
train.loc[
    train["prompt_length"].idxmin(),
    "prompt"
]

In [ ]:
encoder = LabelEncoder()

train["label"] = encoder.fit_transform(train["answer"])

In [ ]:
def build_text(row):
    return (
        row["prompt"]
        + " [SEP] "
        + row["A"]
        + " [SEP] "
        + row["B"]
        + " [SEP] "
        + row["C"]
        + " [SEP] "
        + row["D"]
        + " [SEP] "
        + row["E"]
    )

train["text"] = train.apply(build_text, axis=1)
test["text"] = test.apply(build_text, axis=1)

In [ ]:
def clean(text):

    text = text.lower()

    text = re.sub(r"[^a-z0-9 ]", " ", text)

    text = re.sub(r"\s+", " ", text)

    return text.strip()

train["text"] = train["text"].apply(clean)
test["text"] = test["text"].apply(clean)

In [ ]:
train_df, valid_df = train_test_split(
    train,
    test_size=0.1,
    stratify=train["label"],
    random_state=42
)

In [ ]:
counter = Counter()

for sentence in train_df["text"]:
    counter.update(sentence.split())

vocab = {
    "<PAD>": 0,
    "<UNK>": 1
}

for word in counter:
    vocab[word] = len(vocab)

print("Vocabulary Size:", len(vocab))

In [ ]:
MAX_LEN = 256

def encode(sentence):

    ids = []

    for word in sentence.split():
        ids.append(vocab.get(word, 1))

    ids = ids[:MAX_LEN]

    while len(ids) < MAX_LEN:
        ids.append(0)

    return ids

In [ ]:
class MCQDataset(Dataset):

    def __init__(self, dataframe):

        self.text = dataframe["text"].tolist()
        self.labels = dataframe["label"].tolist()

    def __len__(self):
        return len(self.text)

    def __getitem__(self, idx):

        x = torch.tensor(
            encode(self.text[idx]),
            dtype=torch.long
        )

        y = torch.tensor(
            self.labels[idx],
            dtype=torch.long
        )

        return x, y

In [ ]:
train_dataset = MCQDataset(train_df)
valid_dataset = MCQDataset(valid_df)

train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True
)

valid_loader = DataLoader(
    valid_dataset,
    batch_size=32,
    shuffle=False
)

In [ ]:
class BiLSTMClassifier(nn.Module):

    def __init__(
        self,
        vocab_size,
        embedding_dim=300,
        hidden_dim=256,
        output_dim=5,
        dropout=0.3
    ):

        super().__init__()

        self.embedding = nn.Embedding(
            num_embeddings=vocab_size,
            embedding_dim=embedding_dim,
            padding_idx=0
        )

        self.lstm = nn.LSTM(
            input_size=embedding_dim,
            hidden_size=hidden_dim,
            batch_first=True,
            bidirectional=True
        )

        self.dropout = nn.Dropout(dropout)

        self.fc = nn.Linear(
            hidden_dim * 2,
            output_dim
        )

    def forward(self, x):

        embedded = self.embedding(x)

        output, (hidden, cell) = self.lstm(embedded)

        hidden = torch.cat((hidden[-2], hidden[-1]), dim=1)

        hidden = self.dropout(hidden)

        logits = self.fc(hidden)

        return logits

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = BiLSTMClassifier(
    vocab_size=len(vocab),
    embedding_dim=300,
    hidden_dim=256,
    output_dim=5,
    dropout=0.3
)

model = model.to(device)

In [ ]:
print(model)

In [ ]:
criterion = nn.CrossEntropyLoss()

In [ ]:
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001
)

In [ ]:

EPOCHS = 10

for epoch in range(EPOCHS):

    
    model.train()

    train_loss = 0
    correct = 0
    total = 0

    for X, y in train_loader:

        
        X = X.to(device)
        y = y.to(device)

        
        optimizer.zero_grad()

       
        outputs = model(X)

        
        loss = criterion(outputs, y)

       
        loss.backward()

        
        optimizer.step()

        
        train_loss += loss.item()

        
        _, predicted = torch.max(outputs, 1)

        total += y.size(0)

        correct += (predicted == y).sum().item()

    train_accuracy = 100 * correct / total

    
    model.eval()

    valid_loss = 0
    correct = 0
    total = 0

    with torch.no_grad():

        for X, y in valid_loader:

            X = X.to(device)
            y = y.to(device)

            outputs = model(X)

            loss = criterion(outputs, y)

            valid_loss += loss.item()

            _, predicted = torch.max(outputs, 1)

            total += y.size(0)

            correct += (predicted == y).sum().item()

    valid_accuracy = 100 * correct / total

    print(f"Epoch [{epoch+1}/{EPOCHS}]")
    print(f"Train Loss     : {train_loss/len(train_loader):.4f}")
    print(f"Train Accuracy : {train_accuracy:.2f}%")
    print(f"Valid Loss     : {valid_loss/len(valid_loader):.4f}")
    print(f"Valid Accuracy : {valid_accuracy:.2f}%")
    print("-"*50)

In [ ]:
class TestDataset(Dataset):

    def __init__(self, dataframe):
        self.text = dataframe["text"].tolist()

    def __len__(self):
        return len(self.text)

    def __getitem__(self, idx):
        x = torch.tensor(
            encode(self.text[idx]),
            dtype=torch.long
        )
        return x

In [ ]:
test_dataset = TestDataset(test)

test_loader = DataLoader(
    test_dataset,
    batch_size=32,
    shuffle=False
)

In [ ]:
model.eval()

predictions = []

with torch.no_grad():

    for X in test_loader:

        X = X.to(device)

        outputs = model(X)

        pred = outputs.argmax(dim=1)

        predictions.extend(pred.cpu().numpy())

In [ ]:
answers = encoder.inverse_transform(predictions)

In [ ]:
submission = pd.DataFrame({
    "Id": test["id"],
    "Prediction": answers
})

submission.to_csv("submission.csv", index=False)

submission.head()

In [ ]:
optimizer = AdamW(
    model.parameters(),
    lr=2e-5,
    weight_decay=0.01
)

In [ ]:
MODEL_NAME = "microsoft/deberta-v3-small"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

In [ ]:
class MultipleChoiceDataset(Dataset):

    def __init__(self, dataframe, tokenizer, max_length=256):

        self.data = dataframe.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):

        row = self.data.iloc[idx]

        question = row["prompt"]

        choices = [
            row["A"],
            row["B"],
            row["C"],
            row["D"],
            row["E"]
        ]

        encoding = self.tokenizer(
            [question] * 5,
            choices,
            truncation=True,
            padding="max_length",
            max_length=self.max_length,
            return_tensors="pt"
        )

        item = {
            "input_ids": encoding["input_ids"],
            "attention_mask": encoding["attention_mask"],
            "labels": torch.tensor(row["label"], dtype=torch.long)
        }

        return item

In [ ]:
train_dataset = MultipleChoiceDataset(
    train_df,
    tokenizer
)

valid_dataset = MultipleChoiceDataset(
    valid_df,
    tokenizer
)

In [ ]:
train_loader = DataLoader(
    train_dataset,
    batch_size=8,
    shuffle=True
)

valid_loader = DataLoader(
    valid_dataset,
    batch_size=8,
    shuffle=False
)

In [ ]:
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

model = AutoModelForMultipleChoice.from_pretrained(
    MODEL_NAME
)

model.to(device)

In [ ]:
num_epochs = 3

for epoch in range(num_epochs):

    model.train()

    train_loss = 0

    for batch in train_loader:

        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        optimizer.zero_grad()

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels
        )

        loss = outputs.loss

        loss.backward()

        optimizer.step()

        train_loss += loss.item()

    avg_train_loss = train_loss / len(train_loader)

    print(f"Epoch {epoch+1}/{num_epochs}")
    print(f"Train Loss: {avg_train_loss:.4f}")

In [ ]:
model.eval()

valid_loss = 0
correct = 0
total = 0

with torch.no_grad():

    for batch in valid_loader:

        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels
        )

        loss = outputs.loss
        logits = outputs.logits

        valid_loss += loss.item()

        preds = torch.argmax(logits, dim=1)

        correct += (preds == labels).sum().item()
        total += labels.size(0)

avg_valid_loss = valid_loss / len(valid_loader)
accuracy = 100 * correct / total

print(f"Validation Loss: {avg_valid_loss:.4f}")
print(f"Validation Accuracy: {accuracy:.2f}%")

In [ ]:
batch = next(iter(train_loader))

print("input_ids:", batch["input_ids"].shape)
print("attention_mask:", batch["attention_mask"].shape)
print("labels:", batch["labels"].shape)